# Step 5 — Approach A (Aug 31 origin) and A-vs-B comparison

**Approach A:** same trained TFT, origin **Aug 31, 2026** (no information after that date), predicting Sep 1 → Dec 31 (n = 122). Because 122 > `output_chunk_length` (101), Darts forecasts **autoregressively**: the last 21 days are produced by feeding the model's own forecasts back in as history.

**Approach B** is the Oct 31-origin run from `04_forecast_new_origin.ipynb`.

Needs: the original `local_train_data/` chunks, `calendar_extended_to_dec31.parquet` (real values through Dec 31 are required here, no padding is possible), and B's output folder.

Because A's Sep 1 → Oct 31 is now in the past, the last section **scores A against actuals** for those months. B cannot be scored there (it saw those actuals).

In [ ]:
import os, json, gc, collections.abc
import numpy as np
import pandas as pd
import torch
from darts import TimeSeries
from darts.models import TFTModel
from pytorch_lightning import Callback

## 1. Config (approach A)

In [ ]:
BASE = r"C:\Users\G0004878\Desktop\TFT_Data\Daily_forecasting_model\Iterations in October\_1_Data preparation"
MODEL_NAME = "daily_tft_negbin_emb_2026-10-09_11_37_06"      # the model you already trained

ORIGIN        = pd.Timestamp("2026-08-31")      # APPROACH A: the original history end
FORECAST_END  = pd.Timestamp("2026-12-31")
KEEP_FROM     = pd.Timestamp("2026-09-01")      # first date written to pred_forecast.parquet
NUM_SAMPLES   = 100                             # NegBin is probabilistic; mean/quantiles come from samples
QUANTILES     = [0.45, 0.55, 0.70, 0.75]        # same ones as your PRED_Q45 ... PRED_Q75 sheet
ALLOW_AUTOREGRESSIVE = True                     # n = 122 > output_chunk_length = 101
SERIES_BATCH  = 1500                            # series per predict() call (lower if GPU memory is tight)

# REFRESHED actuals written by 01b_data_refresh_to_oct31.ipynb (same chunk layout as Step 1, last 365 days to ORIGIN).
# Columns needed: time_col, group_col, target_col and every static covariate.
ACTUALS_DIR   = os.path.join(BASE, "local_train_data")     # ORIGINAL chunks (history to Aug 31)
# Calendar written by 01b_data_refresh_to_oct31.ipynb, extended to FORECAST_END.
CALENDAR_PATH = os.path.join(BASE, "calendar_extended_to_dec31.parquet")
OUT_DIR       = os.path.join(BASE, "forecast_approach_A_aug31_origin")

# for the comparison section (read-only)
B_DIR           = os.path.join(BASE, "forecast_new_origin")          # approach B output (Oct 31 origin)
NEW_ACTUALS_DIR = os.path.join(BASE, "local_train_data_refresh")     # actuals through Oct 31 (from 01b)

## 2. Load Model

In [ ]:
# The checkpoint pickled your callbacks; a fresh kernel needs a class with the same name.
class TBDiagnostics(Callback):
    def __init__(self, *a, **k): pass

try:                                   # same module you used when training
    import tft_checks                  # noqa: F401  (EmbeddingGate must be importable)
except ImportError:
    pass


def load_model():
    m = TFTModel.load_from_checkpoint(MODEL_NAME, best=True, map_location="cpu")
    return m

## 3. Build Inputs

In [ ]:
def load_roles():
    with open(os.path.join(BASE, "column_roles.json")) as f:
        roles = json.load(f)
    with open(os.path.join(BASE, roles["vocab_path"])) as f:
        vocab = json.load(f)
    return roles, vocab

In [ ]:
def _pad_calendar(cal, time_col, fut, last_needed):
    """Darts needs future covariates up to ORIGIN + output_chunk_length even when n is smaller.
    Dates after FORECAST_END are only decoder padding: their predictions are discarded.
    Deterministic calendar features are computed exactly; festival flags are set to 0."""
    last = cal[time_col].max()
    if last >= last_needed:
        return cal
    idx = pd.date_range(last + pd.Timedelta(days=1), last_needed, freq="D")
    pad = pd.DataFrame(0.0, index=range(len(idx)), columns=fut)
    pad.insert(0, time_col, idx)
    dt = pad[time_col].dt
    if "DOW_SIN" in fut: pad["DOW_SIN"] = np.sin(2 * np.pi * dt.dayofweek / 7)
    if "DOW_COS" in fut: pad["DOW_COS"] = np.cos(2 * np.pi * dt.dayofweek / 7)
    if "IS_MONTH_END" in fut: pad["IS_MONTH_END"] = (dt.day >= dt.days_in_month - 2).astype(float)
    if "IS_MONTH_START" in fut: pad["IS_MONTH_START"] = (dt.day <= 3).astype(float)
    if "DAYS_TO_MONTH_END" in fut: pad["DAYS_TO_MONTH_END"] = (dt.days_in_month - dt.day) / 31.0
    print(f"  calendar padded {idx[0].date()} -> {idx[-1].date()} ({len(idx)} days; decoder padding only, not forecast)")
    return pd.concat([cal, pad], ignore_index=True)

In [ ]:
def load_calendar(roles, input_len, n, out_len):
    time_col, fut = roles["time_col"], roles["future_covariates"]
    cal = pd.read_parquet(CALENDAR_PATH)
    cal[time_col] = pd.to_datetime(cal[time_col])
    cal = cal.sort_values(time_col).reset_index(drop=True)
    need_from = ORIGIN - pd.Timedelta(days=input_len - 1)
    need_to = ORIGIN + pd.Timedelta(days=n)                  # real values needed up to FORECAST_END
    assert cal[time_col].min() <= need_from and cal[time_col].max() >= need_to, (
        f"Calendar spans {cal[time_col].min().date()} -> {cal[time_col].max().date()} but must span "
        f"{need_from.date()} -> {need_to.date()}.")
    missing = [c for c in fut if c not in cal.columns]
    assert not missing, f"calendar is missing future covariates: {missing}"
    assert not cal[fut].isna().any().any(), "NaN in future covariates"
    cal = _pad_calendar(cal, time_col, fut, ORIGIN + pd.Timedelta(days=max(n, out_len)))   # Darts: end >= origin + output_chunk_length
    return TimeSeries.from_dataframe(cal, time_col=time_col, value_cols=fut, freq=roles["freq"]).astype(np.float32)

In [ ]:
def iter_series(roles, vocab, input_len):
    """Yield (key, TimeSeries) holding exactly the last `input_len` days ending on ORIGIN."""
    import glob
    time_col, group_col, target_col = roles["time_col"], roles["group_col"], roles["target_col"]
    static = roles["static_covariates"]
    code_maps = {c: {v: i for i, v in enumerate(vocab[c])} for c in static}
    first_needed = ORIGIN - pd.Timedelta(days=input_len - 1)
    files = sorted(glob.glob(os.path.join(ACTUALS_DIR, "chunk_*.parquet")))
    assert files, f"no chunk_*.parquet in {ACTUALS_DIR}"
    for path in files:
        df = pd.read_parquet(path)
        df[time_col] = pd.to_datetime(df[time_col])
        df = df[(df[time_col] >= first_needed) & (df[time_col] <= ORIGIN)]
        for key, g in df.groupby(group_col, sort=False):
            g = g.sort_values(time_col)
            t = g[time_col]
            assert len(g) == input_len, f"{key}: {len(g)} days, need {input_len}"
            assert t.iloc[0] == first_needed and t.iloc[-1] == ORIGIN, \
                f"{key}: spans {t.iloc[0].date()} -> {t.iloc[-1].date()}, need {first_needed.date()} -> {ORIGIN.date()}"
            assert (t.diff().dropna() == pd.Timedelta(days=1)).all(), f"{key}: gaps in dates"
            y = g[target_col].to_numpy(dtype=np.float32)
            assert np.isfinite(y).all() and (y >= 0).all(), f"{key}: bad target values"
            codes = [code_maps[c].get(str(g[c].iloc[0]), 0) for c in static]   # 0 = __UNKNOWN__
            ts = TimeSeries.from_times_and_values(
                pd.DatetimeIndex(t), y.reshape(-1, 1), columns=[target_col],
                static_covariates=pd.DataFrame([codes], columns=static, dtype=np.float32))
            yield str(key), ts
        del df
        gc.collect()

## 4. Predict

In [ ]:
def predict_batch(model, batch_series, cov, n, num_samples):
    preds = model.predict(
        n=n, series=batch_series, future_covariates=[cov] * len(batch_series),
        num_samples=num_samples, verbose=False)
    if not isinstance(preds, list):
        preds = [preds]
    out = []
    for p in preds:
        v = p.all_values(copy=False)[:, 0, :]               # (n, samples)
        out.append((p.time_index, v.mean(axis=1), np.quantile(v, QUANTILES, axis=1)))
    return out

In [ ]:
def run(model=None):
    os.makedirs(OUT_DIR, exist_ok=True)
    roles, vocab = load_roles()
    input_len = roles["input_chunk_length"]
    out_len = roles["output_chunk_length"]
    n = (FORECAST_END - ORIGIN).days
    assert n > 0 and (n <= out_len or ALLOW_AUTOREGRESSIVE), f"n={n} must be <= output_chunk_length={out_len}; move the origin later"
    if n > out_len:
        print(f"  NOTE: n={n} > output_chunk_length={out_len}: Darts forecasts autoregressively (approach A).")
    print(f"origin {ORIGIN.date()} | predict {(ORIGIN + pd.Timedelta(days=1)).date()} -> {FORECAST_END.date()} (n={n}) "
          f"| keep from {KEEP_FROM.date()}")

    cov = load_calendar(roles, input_len, n, out_len)
    model = model or load_model()

    rows, batch, keys, done = [], [], [], 0

    def flush():
        nonlocal rows, batch, keys, done
        if not batch:
            return
        for k, (idx, mean, q) in zip(keys, predict_batch(model, batch, cov, n, NUM_SAMPLES)):
            d = pd.DataFrame({roles["group_col"]: k, roles["time_col"]: idx, "PRED_MEAN": mean})
            for qi, qq in enumerate(QUANTILES):
                d[f"PRED_Q{int(qq * 100)}"] = q[qi]
            rows.append(d)
        done += len(batch)
        print(f"  predicted {done:,} series")
        batch, keys = [], []

    for key, ts in iter_series(roles, vocab, input_len):
        batch.append(ts)
        keys.append(key)
        if len(batch) == SERIES_BATCH:
            flush()
    flush()

    full = pd.concat(rows, ignore_index=True)
    assert full["PRED_MEAN"].notna().all(), "NaN in predictions"
    full.to_parquet(os.path.join(OUT_DIR, "pred_full_horizon.parquet"), index=False)
    keep = full[full[roles["time_col"]] >= KEEP_FROM]
    keep.to_parquet(os.path.join(OUT_DIR, "pred_forecast.parquet"), index=False)

    daily = full.groupby(roles["time_col"])["PRED_MEAN"].sum()
    daily.to_csv(os.path.join(OUT_DIR, "daily_total_mean.csv"))
    print(f"\n{KEEP_FROM.date()} -> {FORECAST_END.date()} total (mean):", round(keep["PRED_MEAN"].sum()))
    print("Daily total, last 10 days:\n", daily.tail(10).round(0).to_string())
    return full, keep

## 5. Run approach A

In [ ]:
full_A, keep_A = run()

## 6. Compare A (Aug 31 origin) with B (Oct 31 origin)

In [ ]:
def load_pred(out_dir):
    return pd.read_parquet(os.path.join(out_dir, "pred_full_horizon.parquet"))


def actuals(actuals_dir, roles, start, end):
    """Daily total and series x month totals of actual sales between start and end."""
    tc, gc, yc = roles["time_col"], roles["group_col"], roles["target_col"]
    daily, sm = [], []
    for path in sorted(glob.glob(os.path.join(actuals_dir, "chunk_*.parquet"))):
        d = pd.read_parquet(path, columns=[tc, gc, yc])
        d[tc] = pd.to_datetime(d[tc])
        d = d[(d[tc] >= start) & (d[tc] <= end)]
        daily.append(d.groupby(tc)[yc].sum())
        d["M"] = d[tc].dt.to_period("M")
        sm.append(d.groupby([gc, "M"])[yc].sum().reset_index())
    return pd.concat(daily).groupby(level=0).sum(), pd.concat(sm, ignore_index=True)


def score_vs_actuals(pred, act_sm, roles, start, end):
    """WMAPE and bias at series x month level for the days in [start, end]."""
    tc, gc, yc = roles["time_col"], roles["group_col"], roles["target_col"]
    p = pred[(pred[tc] >= start) & (pred[tc] <= end)].copy()
    p["M"] = p[tc].dt.to_period("M")
    ps = p.groupby([gc, "M"])["PRED_MEAN"].sum().reset_index()
    m = ps.merge(act_sm, on=[gc, "M"])
    rows = []
    for mon, g in list(m.groupby("M")) + [("ALL", m)]:
        a, f = g[yc].to_numpy(float), g["PRED_MEAN"].to_numpy(float)
        rows.append(dict(month=str(mon), series=len(g), actual=a.sum(), forecast=f.sum(),
                         WMAPE=np.abs(f - a).sum() / a.sum(), bias=f.sum() / a.sum() - 1))
    return pd.DataFrame(rows)


def compare_totals(A, B, roles, windows):
    """Daily totals of A and B and their ratio over named windows (label -> (start, end))."""
    tc = roles["time_col"]
    da, db = A.groupby(tc)["PRED_MEAN"].sum(), B.groupby(tc)["PRED_MEAN"].sum()
    d = pd.concat([da.rename("A_aug31_origin"), db.rename("B_oct31_origin")], axis=1).dropna()
    d["B_over_A"] = d["B_oct31_origin"] / d["A_aug31_origin"]
    rows = []
    for lab, (s, e) in windows.items():
        x = d[(d.index >= s) & (d.index <= e)]
        rows.append(dict(window=lab, days=len(x), A=x["A_aug31_origin"].sum(), B=x["B_oct31_origin"].sum(),
                         B_over_A=x["B_oct31_origin"].sum() / x["A_aug31_origin"].sum()))
    return d, pd.DataFrame(rows)


def compare_series(A, B, roles, start, end):
    """Per-series totals for [start, end]: how far apart are A and B, and do they rank series alike?"""
    tc, gc = roles["time_col"], roles["group_col"]
    f = lambda X: X[(X[tc] >= start) & (X[tc] <= end)].groupby(gc)["PRED_MEAN"].sum()
    m = pd.concat([f(A).rename("A"), f(B).rename("B")], axis=1).dropna()
    big = m[(m[["A", "B"]].max(axis=1) >= 1)]          # ignore series with < 1 unit in both
    ratio = (big["B"] / big["A"].replace(0, np.nan)).dropna()
    out = dict(series_compared=len(big),
               spearman_rank_corr=float(big["A"].corr(big["B"], method="spearman")),
               ratio_q05=float(ratio.quantile(.05)), ratio_q25=float(ratio.quantile(.25)),
               ratio_median=float(ratio.median()), ratio_q75=float(ratio.quantile(.75)),
               ratio_q95=float(ratio.quantile(.95)),
               share_differ_gt_25pct=float(((ratio > 1.25) | (ratio < 0.8)).mean()))
    return out, m

In [ ]:
roles, _ = load_roles()
A = load_pred(OUT_DIR)
B = load_pred(B_DIR)
T = lambda s: pd.Timestamp(s)

windows = {
    "Nov 1 - Nov 30": (T("2026-11-01"), T("2026-11-30")),
    "Dec 1 - Dec 10": (T("2026-12-01"), T("2026-12-10")),
    "Dec 11 - Dec 31": (T("2026-12-11"), T("2026-12-31")),
    "Nov 1 - Dec 31": (T("2026-11-01"), T("2026-12-31")),
}
daily, tab = compare_totals(A, B, roles, windows)
print("Totals (sum of PRED_MEAN over all series). B_over_A > 1 means the newer origin forecasts MORE.")
print(tab.round(3).to_string(index=False))

In [ ]:
stats, per_series = compare_series(A, B, roles, T("2026-12-11"), T("2026-12-31"))
print("Dec 11-31, series by series (ratio = B / A):")
for k, v in stats.items():
    print(f"  {k:24s} {v:,.3f}")

### A scored on months that have since happened (Sep 1 → Oct 31, 2026)
WMAPE and bias at series × month level. Bias is Σforecast / Σactual − 1, so −0.30 means A under-forecast by 30%.

In [ ]:
act_daily, act_sm = actuals(NEW_ACTUALS_DIR, roles, T("2026-09-01"), T("2026-10-31"))
print(score_vs_actuals(A, act_sm, roles, T("2026-09-01"), T("2026-10-31")).round(3).to_string(index=False))

a_daily = A.groupby(roles["time_col"])["PRED_MEAN"].sum()
cmp = pd.concat([a_daily.rename("A_forecast"), act_daily.rename("actual")], axis=1).dropna()
cmp["A_over_actual"] = cmp["A_forecast"] / cmp["actual"]
print("\nWeekly A_forecast / actual:")
print(cmp.resample("W").sum().assign(ratio=lambda d: d.A_forecast / d.actual).round(2).to_string())

### Context: last year's actuals on the same calendar dates
Diwali was Oct 20, 2025 and is Nov 8, 2026, so Dec 11-31 is a different distance after Diwali in the two years. This is context only, not a like-for-like benchmark.

In [ ]:
ly_daily, _ = actuals(NEW_ACTUALS_DIR, roles, T("2025-12-11"), T("2025-12-31"))
ly = ly_daily.sum()
for name, X in [("A", A), ("B", B)]:
    s = X[(X[roles["time_col"]] >= T("2026-12-11")) & (X[roles["time_col"]] <= T("2026-12-31"))]["PRED_MEAN"].sum()
    print(f"{name}: Dec 11-31, 2026 = {s:,.0f}  | vs Dec 11-31, 2025 actual {ly:,.0f}: {s / ly:.2f}x")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(daily.index, daily["A_aug31_origin"], label="A: Aug 31 origin")
ax.plot(daily.index, daily["B_oct31_origin"], label="B: Oct 31 origin")
ax.axvline(T("2026-12-11"), color="grey", ls=":")
ax.set_title("Daily total forecast, Nov 1 - Dec 31"); ax.legend(); fig.autofmt_xdate(); plt.show()